<a href="https://colab.research.google.com/github/giggsy1106/DATA-622-NLP-Homework-files/blob/main/HW8_NLP_KOTA_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 nltk scikit-learn textblob spacy
!python -m nltk.downloader punkt stopwords vader_lexicon
!python -m spacy download en_core_web_sm

<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 99.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import json

def fix_notebook(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    # Remove the problematic widgets metadata
    if "metadata" in nb and "widgets" in nb["metadata"]:
        del nb["metadata"]["widgets"]

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=2)




In [3]:
# ========= SETUP =========
!pip install -q transformers sentence-transformers torch nltk yake

import requests
from bs4 import BeautifulSoup
import nltk
nltk.download('punkt')
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer, util
import yake
import re

# URLs from homework
url1 = "https://www.nytimes.com/2024/05/14/climate/climate-change-extreme-weather.html"
url2 = "https://www.foxnews.com/science/climate-change-weather-impact-explained"

# ========= HELPERS =========
def fetch_article_text(url):
    """Download page and return main text (very simple baseline)."""
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")
    # remove script/style
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    # NYT & Fox often put article text in <p>
    paras = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    text = " ".join(paras)
    # basic cleanup
    text = re.sub(r"\s+", " ", text).strip()
    return text

text1 = fetch_article_text(url1)
text2 = fetch_article_text(url2)

print("NYT chars:", len(text1))
print("Fox chars:", len(text2))

# ========= 1. SIMILARITY WITH EMBEDDINGS =========
model = SentenceTransformer("all-MiniLM-L6-v2")

emb1 = model.encode(text1, convert_to_tensor=True)
emb2 = model.encode(text2, convert_to_tensor=True)

cos_sim = util.cos_sim(emb1, emb2).item()
print(f"\nSemantic similarity (cosine): {cos_sim:.3f}")

# ========= 2. TOPICS, SENTIMENT, EMOTIONS =========

# 2a. Very rough topic cues: top TF words (not stopwords)
from collections import Counter
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words("english"))

def top_content_words(text, n=15):
    words = re.findall(r"[A-Za-z']+", text.lower())
    words = [w for w in words if w not in stop_words and len(w) > 3]
    counts = Counter(words)
    return counts.most_common(n)

print("\nApprox topic words NYT:")
print(top_content_words(text1, 15))
print("\nApprox topic words Fox:")
print(top_content_words(text2, 15))

# 2b. Sentiment with VADER
sia = SentimentIntensityAnalyzer()

sent1 = sia.polarity_scores(text1)
sent2 = sia.polarity_scores(text2)

print("\nSentiment NYT:", sent1)
print("Sentiment Fox:", sent2)

# 2c. Simple emotion proxy using chosen lexicons (anger, fear, joy, sadness)
emotion_lex = {
    "fear": ["fear", "threat", "danger", "risk", "catastrophic", "disaster", "devastating"],
    "anger": ["angry", "outrage", "blame", "failure", "negligent"],
    "sadness": ["loss", "tragic", "suffer", "victim", "damage", "destroyed", "death"],
    "joy": ["hope", "progress", "solution", "opportunity", "benefit", "improve"],
}

def emotion_counts(text):
    text_low = text.lower()
    counts = {}
    for emo, words in emotion_lex.items():
        c = 0
        for w in words:
            c += text_low.count(w)
        counts[emo] = c
    return counts

emo1 = emotion_counts(text1)
emo2 = emotion_counts(text2)

print("\nEmotion counts NYT:", emo1)
print("Emotion counts Fox:", emo2)

# ========= 3. TOP 5 KEYWORDS (YAKE) =========
kw_extractor = yake.KeywordExtractor(lan="en", n=1, top=5)

def get_keywords(text, label):
    keywords = kw_extractor.extract_keywords(text)
    # keywords is list of (kw, score) with lower score = more important
    keywords_sorted = sorted(keywords, key=lambda x: x[1])[:5]
    print(f"\nTop 5 keywords for {label}:")
    for kw, score in keywords_sorted:
        print(f"  {kw} (score={score:.4f})")
    return [k for k, _ in keywords_sorted]

nyt_keywords = get_keywords(text1, "NYT")
fox_keywords = get_keywords(text2, "Fox")

# ========= 4. LLM-STYLE SUMMARY TEMPLATE =========
print("\n-------- SUMMARY DRAFT (EDIT IN YOUR OWN WORDS) --------\n")

summary = f"""
1. Similarity:
   - Using sentence-transformer embeddings (all-MiniLM-L6-v2), the cosine similarity
     between the NYT and Fox News articles is approximately {cos_sim:.3f},
     indicating they cover related themes around climate change and extreme weather,
     but are not textually identical.

2. Topics:
   - NYT frequent content words include examples like {top_content_words(text1, 5)}.
   - Fox frequent content words include examples like {top_content_words(text2, 5)}.
   - Both emphasize climate change and extreme weather, but differ in specific
     narrative focus and framing.

3. Sentiment:
   - NYT VADER sentiment scores: {sent1}.
   - Fox VADER sentiment scores: {sent2}.
   - This suggests differences/similarities in overall polarity and intensity
     (you should interpret whether they are more neutral, negative, etc.).

4. Emotions (lexicon-based counts, not a full model):
   - NYT emotion counts: {emo1}.
   - Fox emotion counts: {emo2}.
   - Both pieces reference risk, damage, and threat, but you can compare
     which article leans more toward fear/sadness vs hope/solutions.

5. Keywords:
   - NYT top keywords (YAKE): {nyt_keywords}.
   - Fox top keywords (YAKE): {fox_keywords}.

Overall, both articles discuss the link between climate change and extreme
weather, but differ in framing, emotional tone, and choice of emphasis. In my
analysis, I describe how their topics overlap, how strongly each conveys risk
versus solutions, and how their keyword sets reflect their editorial focus.
"""

print(summary)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 12.5 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


NYT chars: 43
Fox chars: 727


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Semantic similarity (cosine): 0.050

Approx topic words NYT:
[('please', 1), ('enable', 1), ('disable', 1), ('blocker', 1)]

Approx topic words Fox:
[('data', 4), ('provided', 4), ('factset', 4), ('material', 2), ('published', 2), ('broadcast', 2), ('rewritten', 2), ('redistributed', 2), ('news', 2), ('network', 2), ('rights', 2), ('reserved', 2), ('quotes', 2), ('displayed', 2), ('real', 2)]

Sentiment NYT: {'neg': 0.0, 'neu': 0.753, 'pos': 0.247, 'compound': 0.3182}
Sentiment Fox: {'neg': 0.033, 'neu': 0.911, 'pos': 0.056, 'compound': 0.1531}

Emotion counts NYT: {'fear': 0, 'anger': 0, 'sadness': 0, 'joy': 0}
Emotion counts Fox: {'fear': 0, 'anger': 0, 'sadness': 0, 'joy': 2}

Top 5 keywords for NYT:
  blocker (score=0.1583)
  enable (score=0.2974)
  disable (score=0.2974)

Top 5 keywords for Fox:
  Factset (score=0.0886)
  provided (score=0.1089)
  LLC (score=0.1148)
  data (score=0.1207)
  FOX (score=0.1303)

-------- SUMMARY DRAFT (EDIT IN YOUR OWN WORDS) --------


1. Similarit

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
